In [1]:
!pip install -q transformers accelerate sentencepiece

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForCausalLM, pipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Test prompts in Turkmen
test_prompts = [
    "Salam, men gije naharlanmak isleýärin, näme maslahat berýärsiň?",
    "Doglan güne sowgat gerek, näme alsam gowy bolar?",
    "Elim agyrýar, näme etmeli?",
]


Device: cuda


In [2]:
# ---------- Pipeline A: NLLB (tuk_Latn -> eng_Latn) -> Qwen2.5 ----------
print("\nLoading NLLB...")
nllb_name = "facebook/nllb-200-distilled-600M"
nllb_tok = AutoTokenizer.from_pretrained(nllb_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_name).to(device)

def translate_tk_to_en(text):
    nllb_tok.src_lang = "tuk_Latn"
    inputs = nllb_tok(text, return_tensors="pt").to(device)
    forced_bos = nllb_tok.convert_tokens_to_ids("eng_Latn")
    out = nllb_model.generate(**inputs, forced_bos_token_id=forced_bos, max_new_tokens=200)
    return nllb_tok.batch_decode(out, skip_special_tokens=True)[0]

def translate_en_to_tk(text):
    nllb_tok.src_lang = "eng_Latn"
    inputs = nllb_tok(text, return_tensors="pt").to(device)
    forced_bos = nllb_tok.convert_tokens_to_ids("tuk_Latn")
    out = nllb_model.generate(**inputs, forced_bos_token_id=forced_bos, max_new_tokens=200)
    return nllb_tok.batch_decode(out, skip_special_tokens=True)[0]

print("Loading Qwen2.5...")
qwen_name = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_pipe = pipeline("text-generation", model=qwen_name, device=0 if device == "cuda" else -1)

def ask_qwen(english_prompt):
    messages = [{"role": "user", "content": english_prompt}]
    out = qwen_pipe(messages, max_new_tokens=200, do_sample=False)
    return out[0]["generated_text"][-1]["content"]



Loading NLLB...


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [3]:
print("Loading mGPT...")
mgpt_name = "ai-forever/mGPT"
mgpt_tok = AutoTokenizer.from_pretrained(mgpt_name)
mgpt_model = AutoModelForCausalLM.from_pretrained(mgpt_name).to(device)

def ask_mgpt(turkmen_prompt):
    inputs = mgpt_tok(turkmen_prompt, return_tensors="pt").to(device)
    out = mgpt_model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    return mgpt_tok.decode(out[0], skip_special_tokens=True)

Loading mGPT...


config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.46k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.89M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.20M [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.45GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: ai-forever/mGPT
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
for prompt in test_prompts:
    print("\n" + "=" * 80)
    print("TURKMEN PROMPT:", prompt)

    en_translation = translate_tk_to_en(prompt)
    print("\n[NLLB translation -> EN]:", en_translation)

    qwen_answer_en = ask_qwen(en_translation)
    print("[Qwen2.5 answer -> EN]:", qwen_answer_en)

    qwen_answer_tk = translate_en_to_tk(qwen_answer_en)
    print("[NLLB translation -> TK]:", qwen_answer_tk)

    mgpt_answer = ask_mgpt(prompt)
    print("\n[mGPT direct answer -> TK]:", mgpt_answer)


TURKMEN PROMPT: Salam, men gije naharlanmak isleýärin, näme maslahat berýärsiň?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[NLLB translation -> EN]: Hello, I want to have a meal. What do you suggest?


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Qwen2.5 answer -> EN]: As an AI language model, I don't have personal preferences or experiences like humans do. However, I can provide some suggestions based on popular dishes and cuisines around the world.

If you're looking for something quick and easy, you might consider trying out a simple stir-fry with your favorite vegetables and protein. If you prefer something more substantial, you could try making a hearty bowl of soup or pasta with a variety of ingredients such as meat, beans, and vegetables.

For those who enjoy international cuisine, there are many delicious options available. Some popular choices include sushi rolls, tacos, pad thai, pizza, and kebabs. 

Ultimately, the best meal is one that satisfies your taste buds and makes you feel happy and satisfied. So, whatever you choose, just make sure it's something you enjoy!
[NLLB translation -> TK]: Men süni manyda gepleýän adam ýaly, öz islegim we täjirim ýok. Emma men dünýäniň dürli künjeginde ýaşaýan adamlaryň iýmit-içim

...


[mGPT direct answer -> TK]: Doglan güne sowgat gerek, näme alsam gowy bolar?!
Täwyläp, öýde şol töýde,
Olaryň aýdylmazy bolsa aýdylmak
Böýdäniň söýdigine garamazdan,
Güýsdäniň ýagdaýynda özüne
Garamazdan şu ýagdaýyň aýdylmak
Bu şuňa garamazdan, buňa garam

TURKMEN PROMPT: Elim agyrýar, näme etmeli?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[NLLB translation -> EN]: What can I do if my hand hurts?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Qwen2.5 answer -> EN]: If your hand hurts, there are several things you can try to alleviate the pain:

1. Rest: Avoid using your hand as much as possible until it is fully healed.

2. Ice: Apply ice to the affected area for 15-20 minutes every few hours during the first 48 hours after injury. This can help reduce swelling and numb soreness.

3. Compression: Wrap your hand in an elastic bandage or compression wrap to help reduce swelling.

4. Elevation: Keep your hand elevated above heart level when resting to help reduce swelling.

5. Pain relief medication: Over-the-counter pain relievers such as ibuprofen or acetaminophen may help relieve pain and inflammation.

6. Gentle stretching exercises: Once your hand has started to heal, gentle stretching exercises can help improve circulation and prevent stiffness.

7. Seek medical attention: If your hand pain persists or worsens, seek medical attention from a healthcare professional who can provide proper diagnosis and treatment.

Remembe